# 11 — Validation in the collaborator's metric (standardized test RMSE)

Replaces the mean-of-ratios reporting (notebooks 07–10 summaries, `panel_by_model_ratio.csv`)
with the collaborator's notebook-16 procedure, reimplemented step for step, so every number
here is **directly comparable with their feature-space tables** (decision 2026-08-27):

1. features = a representation of each chip (980-d raw latent, chip-mean 5, quantile 100,
   Gram 15, combined 115), extracted once for all 60 sites, P01–P10;
2. standardization: pooled mean/SD per dimension over ALL sites' TRAINING periods only
   (their `get_training_scaler`, ddof=1, SD 0→1);
3. weights: simplex SCM per treated site (SLSQP, their exact settings), on the stacked
   standardized period × dimension observations, **missing observations dropped row-wise**
   (their `valid` mask — this is why site 07's S1 group keeps all 10 sites);
4. expanding design: fit P01–P08 → test P09; refit P01–P09 → test P10;
5. metric: per-site standardized test RMSE, then the plain mean over the 10 treated
   sites; training RMSE of the same fit reported alongside.

Why the old ratio was retired: mean-of-ratios lets small-denominator sites dominate, is
incomparable with anyone else's numbers, and its denominator moves when the representation
changes. The "do donors beat own history" question survives as comparison C2 (per-site
pass counts), not as the headline metric.

In [1]:
import sys
import numpy as np, pandas as pd
from scipy.optimize import minimize

sys.path.insert(0, ".")
import panel_lib as pl
import panel_repr as pr

pd.set_option("display.width", 200)
CACHES = {"chipmean": "latents_biweekly.npz", "genfill": "latents_biweekly_genfill.npz"}
REPRS = ["latent980", "chip_mean", "quantile", "gram", "combined"]
PANELS = {c: pl.Panel.from_npz(pl.LATD / n) for c, n in CACHES.items()}
VAL = pr.load_validity()
ALL_SITES = sorted(PANELS["chipmean"].roster["site_id"])
DON = pr.matched_donors(PANELS["chipmean"])
TREAT = PANELS["chipmean"].treatments


def collab_scm(y, X):
    valid = np.isfinite(y) & np.isfinite(X).all(axis=1)
    y_, X_ = y[valid], X[valid]
    J = X.shape[1]
    obj = lambda w: float(np.mean((y_ - X_ @ w) ** 2))
    res = minimize(obj, np.repeat(1.0 / J, J), method="SLSQP",
                   bounds=[(0.0, 1.0)] * J,
                   constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}],
                   options={"maxiter": 5000, "ftol": 1e-12})
    assert res.success, res.message
    return res.x


def features(panel, sensor, name, arm="plain"):
    """{(site, seq): vector} for P01-P10; NaN vector = missing chip (row-dropped later).
    arm 'masked': pool valid parcels only where >=MIN_VALID, else all-196 fallback.
    arm 'maskdrop': below MIN_VALID the chip is treated as MISSING (NaN vector)."""
    M = 980 if name == "latent980" else pr.dim_of(name)
    F = {}
    for s in ALL_SITES:
        for q in range(1, 11):
            v = panel.L(s, sensor, q)
            if v is None:
                F[(s, q)] = np.full(M, np.nan); continue
            if name == "latent980":
                F[(s, q)] = v; continue
            A = pr._A(panel, s, sensor, q)
            if arm == "plain":
                F[(s, q)] = pr._repr_raw(A, name); continue
            nv = pr.n_valid(VAL, panel, s, sensor, q)
            ok = nv is not None and nv >= pr.MIN_VALID
            if ok:
                F[(s, q)] = pr._repr_masked(A, name, VAL[(s, sensor, panel.pid_of_seq[q])] >= pr.PARCEL_THR)
            elif arm == "masked":
                F[(s, q)] = pr._repr_raw(A, name)
            else:
                F[(s, q)] = np.full(pr.dim_of(name), np.nan)
    return F


def validate(F):
    """their notebook-16 loop -> per-site rows for P09 and P10."""
    rows = []
    for test_q, train_p in ((9, range(1, 9)), (10, range(1, 10))):
        T = np.stack([F[(s, q)] for s in ALL_SITES for q in train_p])
        mu = np.nanmean(T, axis=0)
        sd = np.nanstd(T, axis=0, ddof=1)
        sd[~np.isfinite(sd) | (sd == 0)] = 1.0
        z = lambda v: (v - mu) / sd
        for t in TREAT:
            dl = DON[t]
            ytr = np.concatenate([z(F[(t, q)]) for q in train_p])
            Xtr = np.column_stack([np.concatenate([z(F[(d, q)]) for q in train_p]) for d in dl])
            w = collab_scm(ytr, Xtr)
            e = ytr - Xtr @ w; e = e[np.isfinite(e)]
            tr = float(np.sqrt(np.mean(e * e)))
            yte, Xte = z(F[(t, test_q)]), np.column_stack([z(F[(d, test_q)]) for d in dl])
            e = yte - Xte @ w; e = e[np.isfinite(e)]
            rows.append({"site": t, "test": f"P{test_q:02d}",
                         "test_rmse": float(np.sqrt(np.mean(e * e))), "train_rmse": tr,
                         "n_test_dims": int(np.isfinite(yte).sum())})
    return pd.DataFrame(rows)

print("machinery ready:", len(ALL_SITES), "sites,", len(TREAT), "treated")

machinery ready: 60 sites, 10 treated


## 1. Full grid — both caches, both sensors, five representations

In [2]:
SITE_ROWS, MEAN_ROWS = [], []
for cache, panel in PANELS.items():
    for sensor in pl.SENSORS:
        for name in REPRS:
            df = validate(features(panel, sensor, name))
            df.insert(0, "cache", cache); df.insert(1, "sensor", sensor); df.insert(2, "repr", name)
            SITE_ROWS.append(df)
            m = df.groupby("test")[["test_rmse", "train_rmse"]].mean()
            for tq, r in m.iterrows():
                MEAN_ROWS.append({"cache": cache, "sensor": sensor, "repr": name, "test": tq,
                                  "mean_test_rmse": r.test_rmse, "mean_train_rmse": r.train_rmse,
                                  "n_sites": int((df.test == tq).sum())})
            print(f"{cache:9s} {sensor:10s} {name:10s} "
                  f"P09 {m.loc['P09','test_rmse']:.3f}  P10 {m.loc['P10','test_rmse']:.3f}")
SITE = pd.concat(SITE_ROWS, ignore_index=True)
MEAN = pd.DataFrame(MEAN_ROWS)
SITE.to_csv("panel_collabstyle_scm_sites.csv", index=False)
MEAN.to_csv("panel_collabstyle_scm_validation.csv", index=False)
print("\nmean standardized test RMSE (train), 10 sites everywhere:")
piv = MEAN.pivot_table(index=["sensor", "cache", "repr"], columns="test",
                       values=["mean_test_rmse", "mean_train_rmse"]).round(3)
print(piv.to_string())
print("\ncollaborator band means, same design (their notebook 16): "
      "S1 0.284 (0.286) / 0.313 (0.298); S2 0.839 (0.587) / 0.342 (0.657)")

chipmean  sentinel1  latent980  P09 1.084  P10 1.087
chipmean  sentinel1  chip_mean  P09 0.603  P10 0.701


chipmean  sentinel1  quantile   P09 0.784  P10 0.887
chipmean  sentinel1  gram       P09 0.692  P10 0.732


chipmean  sentinel1  combined   P09 0.776  P10 0.874
chipmean  sentinel2  latent980  P09 1.107  P10 1.251
chipmean  sentinel2  chip_mean  P09 0.577  P10 0.395


chipmean  sentinel2  quantile   P09 0.797  P10 0.581
chipmean  sentinel2  gram       P09 0.625  P10 0.466


chipmean  sentinel2  combined   P09 0.780  P10 0.570
genfill   sentinel1  latent980  P09 1.085  P10 1.088
genfill   sentinel1  chip_mean  P09 0.586  P10 0.714


genfill   sentinel1  quantile   P09 0.752  P10 0.841
genfill   sentinel1  gram       P09 0.669  P10 0.715


genfill   sentinel1  combined   P09 0.745  P10 0.830
genfill   sentinel2  latent980  P09 0.996  P10 1.105


genfill   sentinel2  chip_mean  P09 0.593  P10 0.646


genfill   sentinel2  quantile   P09 0.708  P10 0.789
genfill   sentinel2  gram       P09 0.584  P10 0.721


genfill   sentinel2  combined   P09 0.694  P10 0.783

mean standardized test RMSE (train), 10 sites everywhere:
                             mean_test_rmse        mean_train_rmse       
test                                    P09    P10             P09    P10
sensor    cache    repr                                                  
sentinel1 chipmean chip_mean          0.603  0.701           0.587  0.596
                   combined           0.776  0.874           0.843  0.844
                   gram               0.692  0.732           0.659  0.665
                   latent980          1.084  1.087           1.093  1.093
                   quantile           0.784  0.887           0.861  0.861
          genfill  chip_mean          0.586  0.714           0.605  0.610
                   combined           0.745  0.830           0.822  0.823
                   gram               0.669  0.715           0.673  0.678
                   latent980          1.085  1.088           1.098  1.097


## 2. Masked-pooling arms (Sentinel-2, chip-mean cache) in the same metric

Notebook 10's masked experiment, re-scored: `plain` = statistics of the filled chips
(as shipped); `masked` = pool valid parcels only (all-196 fallback below 10 valid);
`maskdrop` = chips below 10 valid parcels become missing observations, handled by the
same row-dropping as any other missing chip — no per-group machinery needed here.

In [3]:
ARM_ROWS = []
panel = PANELS["chipmean"]
for arm in ("plain", "masked", "maskdrop"):
    for name in ("chip_mean", "quantile", "gram", "combined"):
        df = validate(features(panel, "sentinel2", name, arm=arm))
        df.insert(0, "arm", arm); df.insert(1, "repr", name)
        ARM_ROWS.append(df)
ARMS = pd.concat(ARM_ROWS, ignore_index=True)
ARMS.to_csv("panel_collabstyle_masked_sites.csv", index=False)
print("mean standardized test RMSE, S2 chip-mean cache:")
print(ARMS.pivot_table(index=["repr", "arm"], columns="test", values="test_rmse")
      .reindex(["chip_mean", "quantile", "gram", "combined"], level=0)
      .reindex(["plain", "masked", "maskdrop"], level=1).round(3).to_string())
print("\nper-site P09, Gram (for the cloud slide):")
g = (ARMS.query("repr == 'gram' and test == 'P09'")
     .pivot_table(index="site", columns="arm", values="test_rmse")
     [["plain", "masked", "maskdrop"]].round(2))
print(g.to_string())

mean standardized test RMSE, S2 chip-mean cache:
test                  P09    P10
repr      arm                   
chip_mean plain     0.577  0.395
          masked    0.643  0.398
          maskdrop  1.224  0.475
quantile  plain     0.797  0.581
          masked    0.778  0.578
          maskdrop  1.096  0.712
gram      plain     0.625  0.466
          masked    0.654  0.457
          maskdrop  1.159  0.547
combined  plain     0.780  0.570
          masked    0.759  0.565
          maskdrop  1.103  0.696

per-site P09, Gram (for the cloud slide):
arm             plain  masked  maskdrop
site                                   
treatment_0001   0.35    0.54      1.14
treatment_0002   0.32    0.45      0.60
treatment_0003   0.35    0.30      0.58
treatment_0004   1.37    1.14      1.97
treatment_0005   0.99    0.79      1.22
treatment_0006   0.57    0.57      1.00
treatment_0007   0.26    0.34      1.12
treatment_0008   0.52    0.56      1.12
treatment_0009   0.32    0.78      1.43
treatm

## 3. The analysis-slide numbers, standardized

Worked example (Sentinel-2, site 03, Gram, chip-mean cache, P10) and the candidate-
predictor decomposition (Sentinel-2, chip-mean representation, P10) — same quantities
as report §6.1–6.2, now in pooled-SD units so they live on the same scale as §1.

In [4]:
panel = PANELS["chipmean"]
F = features(panel, "sentinel2", "gram")
train_p = range(1, 10)
T = np.stack([F[(s, q)] for s in ALL_SITES for q in train_p])
mu, sd = np.nanmean(T, axis=0), np.nanstd(T, axis=0, ddof=1)
sd[~np.isfinite(sd) | (sd == 0)] = 1.0
z = lambda v: (v - mu) / sd
t = "treatment_0003"; dl = DON[t]
ytr = np.concatenate([z(F[(t, q)]) for q in train_p])
Xtr = np.column_stack([np.concatenate([z(F[(d, q)]) for q in train_p]) for d in dl])
w = collab_scm(ytr, Xtr)
rms = lambda e: float(np.sqrt(np.mean(e[np.isfinite(e)] ** 2)))
y10 = z(F[(t, 10)]); X10 = np.column_stack([z(F[(d, 10)]) for d in dl])
own = np.nanmean(np.stack([z(F[(t, q)]) for q in range(1, 9)]), axis=0)
print("worked example, site 03 Gram P10 (standardized):")
print(f"  SCM fitted weights        : {rms(y10 - X10 @ w):.3f}")
print(f"  same weights, train window: {rms(ytr - Xtr @ w):.3f}")
print(f"  no donors (own P01-08 avg): {rms(y10 - own):.3f}")
print(f"  plain average of 5 donors : {rms(y10 - X10.mean(axis=1)):.3f}")

Fc = features(panel, "sentinel2", "chip_mean")
Tc = np.stack([Fc[(s, q)] for s in ALL_SITES for q in train_p])
muc, sdc = np.nanmean(Tc, axis=0), np.nanstd(Tc, axis=0, ddof=1)
sdc[~np.isfinite(sdc) | (sdc == 0)] = 1.0
zc = lambda v: (v - muc) / sdc
preds = {"own P01-08 average": [], "donors' P01-08 average": [],
         "matched donors at P10 itself": [], "all 50 controls at P10": []}
ctrl = panel.controls
for t in TREAT:
    y = zc(Fc[(t, 10)])
    dl = DON[t]
    preds["own P01-08 average"].append(rms(y - np.nanmean(
        np.stack([zc(Fc[(t, q)]) for q in range(1, 9)]), axis=0)))
    preds["donors' P01-08 average"].append(rms(y - np.nanmean(
        np.stack([zc(Fc[(d, q)]) for d in dl for q in range(1, 9)]), axis=0)))
    preds["matched donors at P10 itself"].append(rms(y - np.nanmean(
        np.stack([zc(Fc[(d, 10)]) for d in dl]), axis=0)))
    preds["all 50 controls at P10"].append(rms(y - np.nanmean(
        np.stack([zc(Fc[(c, 10)]) for c in ctrl]), axis=0)))
print("\ncandidate predictors of the P10 outcome (S2 chip-mean, standardized, mean over 10 sites):")
for k, v in preds.items():
    print(f"  {k:30s}: {np.mean(v):.3f}")

worked example, site 03 Gram P10 (standardized):
  SCM fitted weights        : 0.169
  same weights, train window: 0.661
  no donors (own P01-08 avg): 1.407
  plain average of 5 donors : 0.188

candidate predictors of the P10 outcome (S2 chip-mean, standardized, mean over 10 sites):
  own P01-08 average            : 0.836
  donors' P01-08 average        : 0.907
  matched donors at P10 itself  : 0.503
  all 50 controls at P10        : 0.452


## Reading

1. **Sentinel-1: the collaborator's band means (0.284 / 0.313) beat every latent
   representation ≈ 2×** (best latent: chip-mean 0.603 / 0.701). The radar level signal
   is real and the latent statistics lose it. Pooling still halves the raw-latent error
   (1.084 → 0.603).
2. **Sentinel-2, clean P10: near-tied** — band means 0.342 vs latent chip-mean 0.395,
   Gram 0.466. **Cloudy P09: every pooled latent beats the band means** (0.577–0.797 vs
   0.839) — but our chips are chip-mean-filled where their band means skip invalid
   pixels, so part of that edge is the shared-fill artifact (see §2 and notebook 10).
3. **Masked arms in comparable units are much milder than the ratio metric suggested.**
   Masking is ≈ cost-free (Gram P09 0.625 → 0.654, P10 0.466 → 0.457) and removes all
   fabricated pixels; the maskdrop arm is uniformly worse (P09 mean 1.16) because
   discarding the P03/P06 chips costs more fit data than the fabricated content costs
   accuracy. The dramatic per-site ratio swings of notebook 10 (site 01 "0.39 → 3.35")
   were mostly the own-history denominator shrinking under masking — real error moved
   only 0.35 → 0.54. This is the concrete argument for retiring the mean-of-ratios.
4. **The §3 decomposition survives the metric change**: donors' history predicts worse
   than the site's own history (0.907 vs 0.836); the pool's value is its observations of
   the predicted fortnight itself (0.503 matched / 0.452 all-50), and fitting adds little
   over a plain donor average (site 03: 0.169 vs 0.188).
